In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
from config import Config
import os
import sys

import numpy as np
from numpy import linalg as LA
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt

from loguru import logger
from scipy.io import loadmat

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using device: {device}")

from loguru import logger

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')

Using device: cuda

4

In [2]:
env_seed = 5        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 5000   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([9, 10, 15, 19, 32, 35, 47, 58, 65, 74, 82, 91, 103, 60]) #11, 36, 75,/ 1,5,9
pp_net = create_123bus()
env = Env_123bus(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
agent_num = env.agentnum
# pf_res_plotly(pp_net);

# Keep extra/non-main experiments in this notebook, but do not run them by default.
RUN_EXTRA_123_EXPERIMENTS = False
RUN_OLD_123_PLOTS = False


D:\Code\Python\Flexible_Voltage_Control\.venv\Lib\site-packages\pandapower\converter\pypower\from_ppc.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  branch_lookup.loc[is_trafo, "element"] = idx_trafo


### Some Plot Function

In [3]:
# plot policy
def plot_policy(policy_net, topology):
    fig, axs = plt.subplots(2, 7, figsize=(15,6))
    title = ['Bus 9', 'Bus 10', 'Bus 15', 'Bus19', 'Bus 32', 'Bus 35', 'Bus 47', 
                'Bus 58', 'Bus 65', 'Bus 74', 'Bus 72', 'Bus 91', 'Bus 103', 'Bus 60']
    for i in range(agent_num):
        axs[i//7][i%7].clear()
        # plot policy
        N = 40
        s_array = np.zeros(N,)
        
        a_array_baseline = np.zeros(N,)
        a_array = np.zeros(N,)
        
        for j in range(N):
            state = torch.tensor([[0.80+0.01*j]])
            s_array[j] = state

            action_baseline = (np.maximum(state.cpu()-1.05, 0)-np.maximum(0.95-state.cpu(), 0)).reshape((1,))
        
            action = policy_net[i](state, topology)
            action = action.detach().cpu().numpy()[0]
            
            a_array_baseline[j] = -action_baseline[0]
            a_array[j] = -action

        axs[i//7][i%7].plot(12*s_array, 50*a_array_baseline, '-.', label = 'Linear')
        axs[i//7][i%7].plot(12*s_array, a_array, label = 'Flexible-DDPG')
        axs[i//7][i%7].set_title(title[i])
        axs[i//7][i%7].legend(loc='lower left')

def plot_safe_net(net):
    fig, axs = plt.subplots(2, 7, figsize=(15,6))
    title = ['Bus 9', 'Bus 10', 'Bus 15', 'Bus19', 'Bus 32', 'Bus 35', 'Bus 47', 
                'Bus 58', 'Bus 65', 'Bus 74', 'Bus 72', 'Bus 91', 'Bus 103', 'Bus 60']
    for i in range(agent_num):
        N = 40
        s_array = np.zeros(N,)
        
        a_array_baseline = np.zeros(N,)
        a_array = np.zeros(N,)
        
        for j in range(N):
            state = np.array([0.8+0.01*j])
            s_array[j] = state

            action_baseline = (np.maximum(state-1.05, 0)-np.maximum(0.95-state, 0)).reshape((1,))
        
            action = net[i].get_action([state])
            
            a_array_baseline[j] = -action_baseline[0]
            a_array[j] = -action

        axs[i//7][i%7].plot(12*s_array, 2*a_array_baseline, '-.', label = 'Linear')
        axs[i//7][i%7].plot(12*s_array, a_array, label = 'Stable-DDPG')
        axs[i//7][i%7].legend(loc='lower left')

def plot_x_policy(policy_net):
    """
    用 Plotly 绘制不同拓扑下的 RLC-FT 策略曲线，所有情形绘制在单个图中
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    N = 400
    for i in range(5):
        policy_vals = []
        # 对于每个拓扑情形，通过 reset_topo 获得对应拓扑设定
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor).detach()
            policy_vals.append(float(-action_tensor.view(-1)[0].cpu()))

        x_vals = np.linspace(10, 14, N)
        fig.add_trace(go.Scatter(x=x_vals, y=policy_vals,
                                 mode='lines',
                                 name=f'Topology {i}'))
    fig.update_layout(
        font=dict(size=16),
        width=700,
        height=500,
        # margin=dict(l=30, r=30, t=30, b=30),   # Uncomment to adjust margins
        margin=dict(r=30,t=30,b=60),
        xaxis_title='Voltage (kV)',
        yaxis_title='Q (MVar)',
        xaxis=dict(
            showgrid=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showgrid=True,
            zeroline=True,
            zerolinecolor='lightgray'
        ),
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,1.0)'
        ),
    )

    fig.show()

### Load model

In [4]:

agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=env.topology_dim, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_123bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2023-09-12/Step_800_Seed_18_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-08-19/Step_950_Seed_2_a{i}.pth'), map_location=device)

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/Stable-DDPG-for-voltage-control/checkpoints/single-phase/123bus/safe-ddpg/policy_net_checkpoint_a{i}.pth', map_location=device)
    

    safe_agent_net[i].load_state_dict(policy_net_dict)

In [5]:
# Old 123-bus policy plotting code retained but skipped by default.
if RUN_OLD_123_PLOTS:
    # plot_policy(agent_policy_net, topology)
    plot_x_policy(agent_policy_net)
    # plot_safe_net(safe_agent_net)


### Flexible NN Contoller

In [6]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []
voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False
    done = False
    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = 0.5 * agent_policy_net[i](torch.tensor(state[i].reshape(1,), dtype=torch.float32, device=device).unsqueeze(0), topology)
            action_agent = action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            # Suppress per-episode success logs for large notebook runs.
            break

        voltage.append(state)
        voltage_episode.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

    if (episode + 1) % 100 == 0 or (episode + 1) == episode_num:
        logger.info('RLC-FT progress: {}/{} episodes, success={}, fail={}', episode + 1, episode_num, len(success_list), len(fail_list))

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

2026-05-09 20:08:21.976 | INFO     | __main__:<module>:78 - RLC-FT progress: 100/5000 episodes, success=100, fail=0


2026-05-09 20:11:07.150 | INFO     | __main__:<module>:78 - RLC-FT progress: 200/5000 episodes, success=200, fail=0


2026-05-09 20:13:57.804 | INFO     | __main__:<module>:78 - RLC-FT progress: 300/5000 episodes, success=300, fail=0


2026-05-09 20:16:44.010 | INFO     | __main__:<module>:78 - RLC-FT progress: 400/5000 episodes, success=400, fail=0


2026-05-09 20:19:23.797 | INFO     | __main__:<module>:78 - RLC-FT progress: 500/5000 episodes, success=500, fail=0


...

2026-05-09 22:23:16.754 | INFO     | __main__:<module>:78 - RLC-FT progress: 4900/5000 episodes, success=4900, fail=0


2026-05-09 22:25:50.198 | INFO     | __main__:<module>:78 - RLC-FT progress: 5000/5000 episodes, success=5000, fail=0


2026-05-09 22:25:50.201 | INFO     | __main__:<module>:80 - total success epsisode is 5000


2026-05-09 22:25:50.203 | INFO     | __main__:<module>:81 - total fail episode is 0


2026-05-09 22:25:50.204 | INFO     | __main__:<module>:82 - number of finished at entire episode is 0


In [7]:
success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
print(np.mean(object_cost))
print(np.std(object_cost))


average recovery step is:

17.6782

12.950190915967225

average reactive power cost is:

272.33044845556356

328.05507485974226

the total cost is:

336.73308227207974

242.41759442786588

In [8]:
# Old extra topology experiment retained but skipped by default.
if RUN_EXTRA_123_EXPERIMENTS:
    # Old extra topology experiment retained but skipped by default.
    if RUN_EXTRA_123_EXPERIMENTS:
        ### test our controller without topology change
        voltage_ = []
        q_ = []
        cost_ = []
        success_list_ = []
        fail_list_ = []
        entire_list_ = []
        control_cost_ = []
        reward_list_ = []

        for episode in range(episode_num):
            state, topology, senario = env.reset_topo(seed=episode)
            topology = 1/env.topology_init
            topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
            last_action = np.zeros((agent_num,1))
            episode_reward = 0
            episode_control = 0
            abnormal_stop = False

            for step in range(step_num):
                action = []
                for i in range(agent_num):
                    action_agent = agent_policy_net[i](torch.tensor(state[i].reshape(1,), dtype=torch.float32, device=device).unsqueeze(0), topology)
                    action_agent = action_agent.detach().cpu().numpy()[0]
                    action.append(action_agent)

                action = last_action - np.asarray(action)
        
                last_action = np.copy(action)
        
                try:
                    next_state, reward, done = env.step(action)
                except pp.powerflow.LoadflowNotConverged:
                    # logger.error(sys.exc_info())
                    logger.error('power flow not converge at epsisode{} step{}', episode, step)
                    fail_list_.append((episode,step))
                    abnormal_stop = True
                    break

                if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage_ violation > 25%, episode ends.
                    logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
                    fail_list_.append((episode,step))
                    abnormal_stop = True
                    break
                if done:
                    success_list_.append((episode,step))
                    # Suppress per-episode success logs for large notebook runs.
                    break

                voltage_.append(state)

                q_.append(action)

                state = next_state
        
                episode_reward += reward
        
                cost_.append(-reward)
        
                episode_control += LA.norm(action, 2)

            reward_list_.append(episode_reward)
            control_cost_.append(episode_control)

            if (not done) and (abnormal_stop == False):
                entire_list_.append(episode)
                logger.info('Episode {} finish with entire step!', episode)

        logger.info('total success epsisode is {}', len(success_list_))
        logger.info('total fail episode is {}', len(fail_list_))
        logger.info('number of finished at entire episode is {}', len(entire_list_))


In [9]:
# Old extra topology summary retained but skipped by default.
if RUN_EXTRA_123_EXPERIMENTS:
    # Old extra topology summary retained but skipped by default.
    if RUN_EXTRA_123_EXPERIMENTS:
        success_list_ = np.array(success_list_)
        print('average recovery step is:')
        print(np.mean(success_list_[:,1]))
        print(np.std(success_list_[:,1]))
        print('average reactive power cost is:')
        print(np.mean(control_cost_))
        print(np.std(control_cost_))


### baseline

In [10]:
### test the base line controller
voltage = []
q = []
cost = []
base_succ_list = []
base_fail_list = []
base_entire_list = []
base_control_cost = []
base_reward_list = []
base_object_cost = []
base_voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False
    done = False

    for step in range(step_num):
        state1 = np.asarray(state-env.vmax)
        state2 = np.asarray(env.vmin-state)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((agent_num,1))
        
        action = (last_action - 30*d_v)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            base_succ_list.append((episode,step))
            # Suppress per-episode success logs for large notebook runs.
            break

        voltage.append(state)
        base_voltage_trajs.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    base_control_cost.append(episode_control)
    base_reward_list.append(episode_reward)
    base_object_cost.append(np.sum(cost))
    
    if (not done) and (abnormal_stop == False):
        base_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

    if (episode + 1) % 100 == 0 or (episode + 1) == episode_num:
        logger.info('Linear progress: {}/{} episodes, success={}, fail={}', episode + 1, episode_num, len(base_succ_list), len(base_fail_list))

logger.info('total success epsisode is {}', len(base_succ_list))
logger.info('total fail episode is {}', len(base_fail_list))
logger.info('number of finished at entire episode is {}', len(base_entire_list))

2026-05-09 22:25:57.283 | INFO     | __main__:<module>:70 - Episode 2 finish with entire step!


2026-05-09 22:26:00.805 | INFO     | __main__:<module>:70 - Episode 3 finish with entire step!


2026-05-09 22:26:06.118 | INFO     | __main__:<module>:70 - Episode 5 finish with entire step!


2026-05-09 22:26:36.439 | INFO     | __main__:<module>:70 - Episode 18 finish with entire step!


2026-05-09 22:27:19.663 | INFO     | __main__:<module>:70 - Episode 39 finish with entire step!


...

2026-05-10 01:31:46.280 | INFO     | __main__:<module>:70 - Episode 4995 finish with entire step!


2026-05-10 01:31:54.567 | INFO     | __main__:<module>:73 - Linear progress: 5000/5000 episodes, success=4543, fail=0


2026-05-10 01:31:54.569 | INFO     | __main__:<module>:75 - total success epsisode is 4543


2026-05-10 01:31:54.570 | INFO     | __main__:<module>:76 - total fail episode is 0


2026-05-10 01:31:54.572 | INFO     | __main__:<module>:77 - number of finished at entire episode is 457


In [11]:
base_succ_list = np.array(base_succ_list)
print('average recovery step is:')
print(np.mean(base_succ_list[:,1]))
print(np.std(base_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(base_control_cost))
print(np.std(base_control_cost))
print('the total cost is:')
print(np.mean(base_object_cost))
print(np.std(base_object_cost))


average recovery step is:

104.97446621175435

41.59371068121441

average reactive power cost is:

2006.034951938549

1777.6912829775658

the total cost is:

2113.2498810776597

969.3491839829636

### Safe DDPG

In [12]:
### test the safe policy net
safe_voltage = []
safe_q = []
safe_cost = []
safe_succ_list = []
safe_fail_list = []
safe_entire_list = []
safe_contorl_cost = []
safe_reward_list = []
safe_object_cost = []
safe_voltage_trajs = []


for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    safe_cost = []
    abnormal_stop = False
    done = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = safe_agent_net[i](torch.tensor([state[i]], dtype=torch.float32, device=device).reshape(1, 1)).detach().cpu().numpy()[0]
            action.append(action_agent)
        
        action = last_action - 10*np.asarray(action).reshape((agent_num, 1))
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            safe_succ_list.append((episode,step))
            # Suppress per-episode success logs for large notebook runs.
            break
        safe_voltage.append(state)
        safe_voltage_trajs.append(state.copy())  # record full state vector

        safe_q.append(action)

        state = next_state
        
        episode_reward += reward
        
        safe_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    safe_contorl_cost.append(episode_control)
    safe_reward_list.append(episode_reward)
    safe_object_cost.append(np.sum(safe_cost))
    if len(safe_voltage) > 0 and senario == 0:
        safe_voltage_trajs.append(np.vstack(safe_voltage))

    if (not done) and (abnormal_stop == False):
        safe_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

    if (episode + 1) % 100 == 0 or (episode + 1) == episode_num:
        logger.info('Safe-DDPG progress: {}/{} episodes, success={}, fail={}', episode + 1, episode_num, len(safe_succ_list), len(safe_fail_list))

logger.info('total success epsisode is {}', len(safe_succ_list))
logger.info('total fail episode is {}', len(safe_fail_list))
logger.info('number of finished at entire episode is {}', len(safe_entire_list))


2026-05-10 01:32:41.426 | INFO     | __main__:<module>:73 - Episode 9 finish with entire step!


2026-05-10 01:32:53.039 | INFO     | __main__:<module>:73 - Episode 10 finish with entire step!


2026-05-10 01:33:09.872 | INFO     | __main__:<module>:73 - Episode 12 finish with entire step!


2026-05-10 01:33:51.694 | INFO     | __main__:<module>:73 - Episode 23 finish with entire step!


2026-05-10 01:35:02.376 | INFO     | __main__:<module>:73 - Episode 43 finish with entire step!


...

2026-05-10 07:17:44.126 | INFO     | __main__:<module>:73 - Episode 4996 finish with entire step!


2026-05-10 07:17:53.921 | INFO     | __main__:<module>:76 - Safe-DDPG progress: 5000/5000 episodes, success=4505, fail=0


2026-05-10 07:17:53.927 | INFO     | __main__:<module>:78 - total success epsisode is 4505


2026-05-10 07:17:53.930 | INFO     | __main__:<module>:79 - total fail episode is 0


2026-05-10 07:17:53.932 | INFO     | __main__:<module>:80 - number of finished at entire episode is 495


In [13]:
safe_succ_list = np.array(safe_succ_list)
print('average recovery step is:')
print(np.mean(safe_succ_list[:,1]))
print(np.std(safe_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(safe_contorl_cost))
print(np.std(safe_contorl_cost))
print('the total cost is:')
print(np.mean(safe_object_cost))
print(np.std(safe_object_cost))

average recovery step is:

52.067480577136514

25.359092465583927

average reactive power cost is:

1116.5783419279455

1131.279046664986

the total cost is:

1225.4694283966508

942.520013661619

In [14]:
# Old 123-bus plotting code retained but skipped by default.
if RUN_OLD_123_PLOTS:
    # Old 123-bus plotting code retained but skipped by default.
    if RUN_OLD_123_PLOTS:
        import plotly.graph_objects as go
        import numpy as np
        from plotly.subplots import make_subplots

        # Calculate statistics from raw data
        methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
        metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

        # Extract data and calculate statistical values
        data = {
            'Voltage Recovery Time': {
                'means': [
                    np.mean(base_succ_list[:,1]),      # Linear
                    np.mean(safe_succ_list[:,1]),      # Safe-DDPG
                    np.mean(success_list[:,1])         # RLC-FT
                ],
                'stds': [
                    np.std(base_succ_list[:,1]),       # Linear
                    np.std(safe_succ_list[:,1]),       # Safe-DDPG
                    np.std(success_list[:,1])          # RLC-FT
                ],
                'unit': 'Steps'
            },
            'Reactive Power': {
                'means': [
                    np.mean(base_control_cost),        # Linear
                    np.mean(safe_contorl_cost),        # Safe-DDPG (note original variable name spelling)
                    np.mean(control_cost)              # RLC-FT
                ],
                'stds': [
                    np.std(base_control_cost),         # Linear
                    np.std(safe_contorl_cost),         # Safe-DDPG
                    np.std(control_cost)               # RLC-FT
                ],
                'unit': 'MVar'
            },
            'Total Objective Cost': {
                'means': [
                    np.mean(base_object_cost),         # Linear
                    np.mean(safe_object_cost),         # Safe-DDPG
                    np.mean(object_cost)               # RLC-FT
                ],
                'stds': [
                    np.std(base_object_cost),          # Linear
                    np.std(safe_object_cost),          # Safe-DDPG
                    np.std(object_cost)                # RLC-FT
                ],
                'unit': ''
            }
        }

        # Print calculated results for verification (including truncation info)
        print("Calculated Statistics:")
        truncation_info = []
        for metric in metrics:
            print(f"\n{metric}:")
            for i, method in enumerate(methods):
                mean_val = data[metric]['means'][i]
                std_val = data[metric]['stds'][i]
                lower_bound = mean_val - std_val
        
                if lower_bound < 0:
                    truncation_info.append(f"{metric} - {method}")
                    print(f"  {method}: {mean_val:.2f} ± {std_val:.2f} (truncated at 0)")
                else:
                    print(f"  {method}: {mean_val:.2f} ± {std_val:.2f}")

        # Nature-style colors
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

        # Create subplots
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=[metric for metric in metrics],  # Only metric names, no units
            horizontal_spacing=0.15
        )

        # Create grouped bar charts for each metric
        for i, metric in enumerate(metrics):
            means = data[metric]['means']
            stds = data[metric]['stds']
    
            for j, method in enumerate(methods):
                mean_val = means[j]
                std_val = stds[j]
        
                # Check if truncation is needed (handle negative values)
                is_truncated = (mean_val - std_val) < 0
        
                # Calculate error bar lengths
                if is_truncated:
                    # Truncation handling: lower error bar only extends to 0
                    error_minus = mean_val  # distance from mean to 0
                    error_plus = std_val    # keep original upper error bar
                else:
                    # Normal case: symmetric error bars
                    error_minus = std_val
                    error_plus = std_val
        
                fig.add_trace(
                    go.Bar(
                        x=[method],
                        y=[mean_val],
                        error_y=dict(
                            type='data',
                            symmetric=False,
                            array=[error_plus],      # upper error bar
                            arrayminus=[error_minus], # lower error bar (may be truncated)
                            visible=True,
                            thickness=3,
                            width=5
                        ),
                        name=method,
                        marker_color=colors[j],
                        marker_line=dict(width=0.8, color='black'),
                        showlegend=(i == 0),
                        width=0.6
                    ),
                    row=1, col=i+1
                )
        
                # Add value labels above error bars
                if i == 0:  #
                    fig.add_annotation(
                        x=method,
                        y=mean_val + max(means) * 0.05,  # above error bars
                        xshift=25,  # shift value labels to the right (in pixels, adjustable)
                        text=f"{mean_val:.1f}",
                        showarrow=False,
                        font=dict(size=14, color='black'),
                        row=1, col=i+1
                    )

                if i == 1:  #
                    fig.add_annotation(
                        x=method,
                        y=mean_val + max(means) * 0.05,  # above error bars
                        xshift=30,  # shift value labels to the right (in pixels, adjustable)
                        text=f"{mean_val:.1f}",
                        showarrow=False,
                        font=dict(size=14, color='black'),
                        row=1, col=i+1
                    )

                if i == 2:  #
                    fig.add_annotation(
                        x=method,
                        y=mean_val + max(means) * 0.05,  # above error bars
                        xshift=40,  # shift value labels to the right (in pixels, adjustable)
                        text=f"{mean_val:.1f}",
                        showarrow=False,
                        font=dict(size=14, color='black'),
                        row=1, col=i+1
                    )
        
                # Add truncation marker at bottom if truncated
                if is_truncated:
                    fig.add_annotation(
                        x=method,
                        y=max(means) * 0.02,  # near bottom
                        text="⊥",  # truncation symbol
                        showarrow=False,
                        font=dict(size=16, color='red'),
                        row=1, col=i+1
                    )

        # Update layout (removed title)
        fig.update_layout(
            width=1200,
            height=500,
            font=dict(
                family="Arial, sans-serif",
                size=16,
                color="black"
            ),
            plot_bgcolor='white',
            paper_bgcolor='white',
            legend=dict(
                x=1.02,
                y=1,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255,255,255,0)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=14)
            ),
            margin=dict(l=70, r=140, t=80, b=70)  # reduced top margin since no title
        )

        # Update axis styles with units on y-axes
        y_axis_titles = [
            data['Voltage Recovery Time']['unit'],
            data['Reactive Power']['unit'], 
            data['Total Objective Cost']['unit']
        ]

        for i in range(1, 4):
            fig.update_xaxes(
                row=1, col=i,
                showgrid=False,
                showline=True,
                linewidth=1.5,
                linecolor='black',
                tickfont=dict(size=14)
            )
    
            # Set y-axis title with unit
            y_title = y_axis_titles[i-1] if y_axis_titles[i-1] else ""
    
            fig.update_yaxes(
                row=1, col=i,
                title=dict(
                    text=y_title,
                    font=dict(size=14, color='black')
                ),
                showgrid=True,
                gridwidth=0.5,
                gridcolor='lightgray',
                showline=True,
                linewidth=1.5,
                linecolor='black',
                tickfont=dict(size=14),
                rangemode='tozero'  # ensure y-axis starts from 0
            )

        # Update subplot title formatting (without units now)
        for annotation in fig['layout']['annotations']:
            annotation['font'] = dict(size=16, color='black')

        # Display figure
        fig.show()

        # Display truncation information
        if truncation_info:
            print(f"\n⊥ Truncation Applied (error bars cut at zero for):")
            for info in truncation_info:
                print(f"  - {info}")
            print("Note: Red ⊥ symbols indicate where error bars were truncated at zero")

        # Calculate performance improvement percentages
        print("\n" + "="*50)
        print("Performance Improvement of RLC-FT vs Linear:")
        recovery_improve = (data['Voltage Recovery Time']['means'][0] - data['Voltage Recovery Time']['means'][2]) / data['Voltage Recovery Time']['means'][0] * 100
        power_improve = (data['Reactive Power']['means'][0] - data['Reactive Power']['means'][2]) / data['Reactive Power']['means'][0] * 100
        cost_improve = (data['Total Objective Cost']['means'][0] - data['Total Objective Cost']['means'][2]) / data['Total Objective Cost']['means'][0] * 100

        print(f"• Voltage Recovery: {recovery_improve:.1f}% faster")
        print(f"• Reactive Power: {power_improve:.1f}% reduction")
        print(f"• Total Cost: {cost_improve:.1f}% reduction")

        # Data integrity check
        print("\n" + "="*50)
        print("Data Quality Summary:")
        for metric in metrics:
            means = data[metric]['means']
            stds = data[metric]['stds']
    
            print(f"\n{metric}:")
            for j, method in enumerate(methods):
                cv = (stds[j] / means[j]) * 100  # coefficient of variation
                truncated = "Yes" if (means[j] - stds[j]) < 0 else "No"
                print(f"  {method}: CV={cv:.1f}%, Truncated={truncated}")

        # Save high-quality images (optional)
        fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.png"), width=1200, height=500, scale=2)
        fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.pdf"), width=1200, height=500)


In [15]:
import json
import os

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 56-bus style comparison figure, generated from the latest 123-bus simulation results.
methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

base_succ_arr = np.asarray(base_succ_list)
safe_succ_arr = np.asarray(safe_succ_list)
rlc_succ_arr = np.asarray(success_list)

data = {
    'Voltage Recovery Time': {
        'means': [
            np.mean(base_succ_arr[:, 1]),
            np.mean(safe_succ_arr[:, 1]),
            np.mean(rlc_succ_arr[:, 1]),
        ],
        'stds': [
            np.std(base_succ_arr[:, 1]),
            np.std(safe_succ_arr[:, 1]),
            np.std(rlc_succ_arr[:, 1]),
        ],
        'unit': 'Steps',
    },
    'Reactive Power': {
        'means': [
            np.mean(base_control_cost),
            np.mean(safe_contorl_cost),
            np.mean(control_cost),
        ],
        'stds': [
            np.std(base_control_cost),
            np.std(safe_contorl_cost),
            np.std(control_cost),
        ],
        'unit': 'MVar',
    },
    'Total Objective Cost': {
        'means': [
            np.mean(base_object_cost),
            np.mean(safe_object_cost),
            np.mean(object_cost),
        ],
        'stds': [
            np.std(base_object_cost),
            np.std(safe_object_cost),
            np.std(object_cost),
        ],
        'unit': '',
    },
}

print("Calculated Statistics:")
truncation_info = []
for metric in metrics:
    print(f"\n{metric}:")
    for i, method in enumerate(methods):
        mean_val = data[metric]['means'][i]
        std_val = data[metric]['stds'][i]
        if mean_val - std_val < 0:
            truncation_info.append(f"{metric} - {method}")
            print(f"  {method}: {mean_val:.2f} +/- {std_val:.2f} (truncated at 0)")
        else:
            print(f"  {method}: {mean_val:.2f} +/- {std_val:.2f}")

colors = {
    'Linear': '#5AA1D6',
    'Safe-DDPG': '#E7904B',
    'RLC-FT': '#53C19E',
}
colors_edge = {
    'Linear': '#2F6E9D',
    'Safe-DDPG': '#B3611E',
    'RLC-FT': '#2D8C73',
}
err_color = 'rgba(0,0,0,0.55)'

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[metric for metric in metrics],
    horizontal_spacing=0.15,
)

for i, metric in enumerate(metrics):
    means = data[metric]['means']
    stds = data[metric]['stds']

    for j, method in enumerate(methods):
        mean_val = means[j]
        std_val = stds[j]
        is_truncated = (mean_val - std_val) < 0
        error_minus = mean_val if is_truncated else std_val
        error_plus = std_val

        fig.add_trace(
            go.Bar(
                x=[method],
                y=[mean_val],
                error_y=dict(
                    type='data',
                    symmetric=False,
                    array=[error_plus],
                    arrayminus=[error_minus],
                    visible=True,
                    thickness=2.5,
                    width=5,
                    color=err_color,
                ),
                name=method,
                marker_color=colors[method],
                marker_line=dict(width=1.6, color=colors_edge[method]),
                showlegend=(i == 0),
                width=0.6,
            ),
            row=1,
            col=i + 1,
        )

        fig.add_annotation(
            x=method,
            y=mean_val + max(means) * 0.1,
            xshift=30,
            text=f"{mean_val:.1f}",
            showarrow=False,
            font=dict(size=16, color='black'),
            row=1,
            col=i + 1,
        )

        if is_truncated:
            fig.add_annotation(
                x=method,
                y=max(means) * 0.02,
                text="⊥",
                showarrow=False,
                font=dict(size=16, color='red'),
                row=1,
                col=i + 1,
            )

fig.update_layout(
    width=1200,
    height=500,
    font=dict(family="Arial, sans-serif", size=20, color="black"),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.6)',
        bordercolor='rgba(0,0,0,0)',
        font=dict(size=18),
    ),
    margin=dict(l=70, r=140, t=80, b=70),
)

y_axis_titles = [
    data['Voltage Recovery Time']['unit'],
    data['Reactive Power']['unit'],
    data['Total Objective Cost']['unit'],
]

for i in range(1, 4):
    fig.update_xaxes(
        row=1,
        col=i,
        showgrid=False,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18),
    )

    fig.update_yaxes(
        row=1,
        col=i,
        title=dict(text=y_axis_titles[i - 1], font=dict(size=18, color='black')),
        showgrid=True,
        gridwidth=0.5,
        gridcolor='lightgray',
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18),
        rangemode='tozero',
    )

for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=20, color='black')

fig.show()

if truncation_info:
    print("\n⊥ Truncation Applied (error bars cut at zero for):")
    for info in truncation_info:
        print(f"  - {info}")
    print("Note: Red ⊥ symbols indicate where error bars were truncated at zero")

print("\n" + "=" * 50)
print("Data Quality Summary:")
for metric in metrics:
    means = data[metric]['means']
    stds = data[metric]['stds']
    print(f"\n{metric}:")
    for j, method in enumerate(methods):
        cv = (stds[j] / means[j]) * 100 if means[j] != 0 else float('nan')
        truncated = "Yes" if (means[j] - stds[j]) < 0 else "No"
        print(f"  {method}: CV={cv:.1f}%, Truncated={truncated}")

output_dir = os.path.join(Config.data_path, "images", "123bus")
os.makedirs(output_dir, exist_ok=True)

fig.write_image(os.path.join(output_dir, "performance_comparison.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(output_dir, "performance_comparison.pdf"), width=1200, height=500)
fig.write_image(os.path.join(output_dir, "performance_comparison_5000.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(output_dir, "performance_comparison_5000.pdf"), width=1200, height=500)


def _method_summary(method_idx, success_arr, fail_list, entire_list):
    return {
        "success_count": int(len(success_arr)),
        "fail_count": int(len(fail_list)),
        "entire_count": int(len(entire_list)),
        "recovery_mean": float(data['Voltage Recovery Time']['means'][method_idx]),
        "recovery_std": float(data['Voltage Recovery Time']['stds'][method_idx]),
        "control_mean": float(data['Reactive Power']['means'][method_idx]),
        "control_std": float(data['Reactive Power']['stds'][method_idx]),
        "object_mean": float(data['Total Objective Cost']['means'][method_idx]),
        "object_std": float(data['Total Objective Cost']['stds'][method_idx]),
    }


summary = {
    "Linear": _method_summary(0, base_succ_arr, base_fail_list, base_entire_list),
    "Safe-DDPG": _method_summary(1, safe_succ_arr, safe_fail_list, safe_entire_list),
    "RLC-FT": _method_summary(2, rlc_succ_arr, fail_list, entire_list),
}

figure_data = {
    metric: {
        "means": [float(v) for v in data[metric]["means"]],
        "stds": [float(v) for v in data[metric]["stds"]],
        "unit": data[metric]["unit"],
    }
    for metric in metrics
}

stats_path = os.path.join(output_dir, "performance_main_5000_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "episodes": int(episode_num),
            "step_num": int(step_num),
            "summary": summary,
            "figure_data": figure_data,
            "truncation_info": truncation_info,
        },
        f,
        indent=2,
    )

print(f"Saved 123-bus 5000-scenario stats to {stats_path}")


Calculated Statistics:


Voltage Recovery Time:

  Linear: 104.97 +/- 41.59

  Safe-DDPG: 52.07 +/- 25.36

  RLC-FT: 17.68 +/- 12.95

...


Total Objective Cost:

  Linear: CV=45.9%, Truncated=No

  Safe-DDPG: CV=76.9%, Truncated=No

  RLC-FT: CV=72.0%, Truncated=No

Saved 123-bus 5000-scenario stats to D:/Code/Python/Flexible_Voltage_Control/images\123bus\performance_main_5000_stats.json